[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/two-phaseflow/JED116F/blob/main/notebooks/01_state_and_motion_of_water.ipynb)

Click the badge to open this notebook in Google Colab. In Colab, choose **File > Save a copy in Drive** before you start, so your work is kept.

# JEÐ116F Notebook 01. The state and motion of water in a geothermal reservoir

Lecture 1, Monday 5 October 2026. We work through this in class. Finish whatever is left during the week, it is the base for the notebooks that follow.

**Cell types**

- **A. Run and look.** The code is complete. Run it and read the output.
- **B. Fill the gaps.** Replace every `___` with code. A test cell checks your answer.
- **C. Extensions.** Open questions for when you finish early.

**Rules for this course**

1. Keep **units** next to every number in your code comments.
2. You may use AI tools for error messages, plotting and coding ideas. You must be able to explain every line you hand in.

**Parts**

1. Setup, units and water properties
2. Enthalpy, heat capacity and two-phase mixtures
3. Pressure with depth and the boiling point with depth curve

## Part 1. Setup, units and water properties

We use the `iapws` package, which implements IAPWS-IF97, the international standard for water and steam properties.

Units in `iapws` are pressure in **MPa**, temperature in **K**, specific enthalpy and internal energy in **kJ/kg**, density in kg/m³ and viscosity in Pa s. In this notebook we give temperature in °C and convert inside the helper functions.

In [ ]:
# Install the water property package.  Runs once per Colab session, takes about 20 seconds.
%pip install -q iapws

# If a later notebook needs data files from the course repository, fetch them like this:
#   !wget -q https://raw.githubusercontent.com/two-phaseflow/JED116F/main/data/<file>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from iapws import IAPWS97, IAPWS95

plt.rcParams.update({"font.size": 13, "figure.figsize": (8, 5), "axes.grid": True})

G = 9.81                   # gravitational acceleration, m/s2
P_ATM = 0.101325           # atmospheric pressure, MPa
K0 = 273.15                # 0 C expressed in K
P_CRIT = IAPWS95.Pc        # critical pressure, MPa
T_CRIT = IAPWS95.Tc - K0   # critical temperature, C
print(f"Critical point: {P_CRIT} MPa, {T_CRIT:.3f} C")

In [ ]:
def water(P, T):
    '''Single-phase water at pressure P (MPa) and temperature T (C).
    Returns rho (kg/m3), u and h (kJ/kg), mu (Pa s), cp (kJ/kg/K) and the phase name.'''
    w = IAPWS97(P=P, T=T + K0)
    return {"rho": w.rho, "u": w.u, "h": w.h, "mu": w.mu, "cp": w.cp, "phase": w.phase}


def saturation(P):
    '''Boiling-curve properties at pressure P (MPa), valid below the critical pressure.
    Returns T (C) and saturated liquid (l) and vapour (v) density, enthalpy and viscosity.'''
    liq = IAPWS97(P=P, x=0)
    vap = IAPWS97(P=P, x=1)
    return {"T": liq.T - K0, "rho_l": liq.rho, "rho_v": vap.rho,
            "h_l": liq.h, "h_v": vap.h, "mu_l": liq.mu, "mu_v": vap.mu}


print(water(10, 300))
print(saturation(1.0))

### B1. Units and a first look at enthalpy

1. Pressure at 1 km depth in a well full of cold water, $P = P_0 + \rho g z$, where $P_0$ is atmospheric pressure (Pa), $\rho$ is density (kg/m³), $g$ is gravitational acceleration (m/s²) and $z$ is depth (m).
2. Energy to heat 1 kg of water from 10 °C to 300 °C at 10 MPa. First with a constant heat capacity, $\Delta h = c_p \Delta T$, where $c_p = 4.2$ kJ/kg/K and $\Delta T$ is the temperature change (K). Then with the steam tables.

In [ ]:
# 1. Pressure at the bottom of a 1000 m cold water column
rho_cold = 1000.0          # kg/m3
depth = 1000.0             # m
P_bottom_MPa = ___         # MPa

# 2. Heating 1 kg from 10 C to 300 C at 10 MPa
dh_cp = ___                # kJ/kg, constant cp = 4.2 kJ/kg/K
dh_table = ___             # kJ/kg, from water()

print(f"P = {P_bottom_MPa:.2f} MPa = {10 * P_bottom_MPa:.1f} bar")
print(f"constant cp: {dh_cp:.0f} kJ/kg, steam tables: {dh_table:.0f} kJ/kg")

In [ ]:
# Test
assert abs(P_bottom_MPa - 9.91) < 0.02, "1 MPa = 1e6 Pa"
assert abs(dh_table - 1291) < 3
print("B1 OK")

## Part 2. Enthalpy, heat capacity and latent heat

Heat one kilogram of water at constant pressure. The first law is $dU = \delta Q - P\,dV$, and at constant pressure $P\,dV = d(PV)$, so

$$(\delta q)_P = dh, \qquad h = u + \frac{P}{\rho}$$

where $(\delta q)_P$ is the heat added per kilogram at constant pressure (J/kg), $h$ is specific enthalpy (J/kg), $u$ is specific internal energy (J/kg), $P$ is pressure (Pa) and $\rho$ is density (kg/m³). Enthalpy is the heat you must add at constant pressure. The term $P/\rho$ is the volume work carried with the fluid.


### B2.1 The energy budget of heating and boiling

In [ ]:
P = 1.0                                   # MPa
cold = water(P, 10)                       # liquid at 10 C
sat = saturation(P)                       # boiling curve at 1 MPa
print(f"boiling temperature at {P} MPa: {sat['T']:.1f} C")

sensible = ___                            # kJ/kg, heating liquid from 10 C to the boiling point
latent = ___                              # kJ/kg, boiling at constant pressure

# How much of each is volume work P dv?  v = 1/rho, so P dv is in kJ/kg when P is in MPa and rho in kg/m3
pdv_liquid = ___                          # kJ/kg
pdv_boiling = ___                         # kJ/kg

print(f"sensible heat {sensible:.0f} kJ/kg (P dv = {pdv_liquid:.2f} kJ/kg)")
print(f"latent heat   {latent:.0f} kJ/kg (P dv = {pdv_boiling:.0f} kJ/kg)")

In [ ]:
# Test
assert abs(sensible - 720) < 3 and abs(latent - 2014) < 3
assert abs(pdv_boiling - 193) < 3, "P dv: keep MPa and kg/m3 consistent"
print("B2.1 OK")

Boiling costs about three times as much energy as heating the liquid all the way from 10 °C, and about 10 % of the latent heat is spent pushing the surroundings aside as the steam expands. That term is why we book-keep with $h$ and not with $u$.

### A. What enthalpy is, and why flow carries it

Enthalpy is defined as

$$h = u + \frac{P}{\rho}$$

where $u$ is specific internal energy (J/kg), $P$ is pressure (Pa), $\rho$ is density (kg/m³) and $h$ is specific enthalpy (J/kg). It is a state property in the same sense as density. Every pressure and temperature has one value of $h$, and you can look it up for any state of water, flowing or not.

Two separate facts make it useful.

The first is the one you met in thermodynamics. At constant pressure, the heat added to a closed system equals its enthalpy change, and that is how enthalpy was measured and tabulated in the first place.

The second is the one that matters in a reservoir. Think of one kilogram crossing a surface, say the feed zone of a well. It brings its internal energy $u$ with it, and while it is crossing, the fluid behind pushes on it and does work on it. One kilogram occupies a volume $1/\rho$, so that work is $P/\rho$ per kilogram. The energy that crosses the surface with each kilogram is therefore $u + P/\rho$, which is $h$. In other words, enthalpy already includes the work needed to push fluid into and out of a control volume, which is why it is the variable used for energy carried by a flow, and why a well test reports enthalpy rather than temperature.

Section B2.4 puts numbers on this for fluid rising in a well.

### B2.2 Heat capacity is the slope of enthalpy

$$c_p = \left(\frac{\partial h}{\partial T}\right)_P, \qquad \Delta h = \int_{T_1}^{T_2} c_p\, dT$$

where $c_p$ is the isobaric specific heat capacity (J/kg/K) and $T$ is temperature (K). Using $\Delta h = c_p \Delta T$ with one value of $c_p$ assumes $c_p$ is constant.

In [ ]:
T_grid_liq = np.linspace(10, 300, 200)                 # C
cp_values = np.array([___ for T in T_grid_liq])        # kJ/kg/K at 10 MPa

dh_constant_cp = 4.2 * (300 - 10)                      # kJ/kg
dh_integral = np.trapezoid(cp_values, T_grid_liq)      # kJ/kg (np.trapz on older numpy)
dh_table = ___                                         # kJ/kg, straight from enthalpy

print(f"constant cp: {dh_constant_cp:.0f} kJ/kg")
print(f"integral of cp: {dh_integral:.0f} kJ/kg")
print(f"from the tables: {dh_table:.0f} kJ/kg")

plt.plot(T_grid_liq, cp_values)
plt.axhline(4.2, ls="--", color="gray")
plt.xlabel("temperature (C)"); plt.ylabel("cp of liquid at 10 MPa (kJ/kg/K)")
plt.show()

In [ ]:
# Test
assert abs(dh_integral - dh_table) < 15, "The integral of cp should reproduce the enthalpy difference"
print(f"constant cp is off by {100 * (dh_constant_cp - dh_table) / dh_table:+.0f} %")
print("B2.2 OK")

### A. The pressure-enthalpy diagram

Inside the two-phase dome, liquid and steam coexist at the same pressure and temperature. The steam mass fraction of a mixture with enthalpy $h$ at pressure $P$ is

$$x = \frac{h - h_l(P)}{h_v(P) - h_l(P)}$$

where $x$ is the steam mass fraction (dimensionless), and $h_l(P)$ and $h_v(P)$ are the specific enthalpies of saturated liquid and saturated vapour at pressure $P$ (kJ/kg). The dashed lines below are lines of constant $x$.

In [ ]:
def ph_diagram(ax=None, isotherms=(100, 200, 300, 350, 400, 500)):
    '''Draw the two-phase dome, lines of constant steam fraction and isotherms on a P-h diagram.'''
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 6))
    P_dome = np.linspace(0.01, P_CRIT * 0.9999, 300)
    sat = [saturation(p) for p in P_dome]
    hl = np.array([s["h_l"] for s in sat]); hv = np.array([s["h_v"] for s in sat])
    ax.plot(hl, P_dome, "k", lw=2); ax.plot(hv, P_dome, "k", lw=2)
    for xq in [0.2, 0.4, 0.6, 0.8]:
        ax.plot(hl + xq * (hv - hl), P_dome, "g--", lw=0.8)
    for T in isotherms:
        P_line = np.linspace(0.02, 40, 400)
        h_line = []
        for p in P_line:
            if p < P_CRIT and abs(saturation(p)["T"] - T) < 1.0:
                h_line.append(np.nan)            # skip the flat two-phase part
            else:
                try:
                    h_line.append(water(p, T)["h"])
                except Exception:
                    h_line.append(np.nan)
        ax.plot(h_line, P_line, lw=1, label=f"{T} C")
    ax.set_xlabel("Specific enthalpy h (kJ/kg)"); ax.set_ylabel("Pressure P (MPa)")
    ax.set_xlim(0, 3600); ax.set_ylim(0, 40)
    return ax


ax = ph_diagram()
ax.legend(title="isotherms", fontsize=10)
plt.show()

### B2.3 Steam mass fraction and steam volume fraction

The steam volume fraction (saturation) is

$$S = \frac{x/\rho_v}{x/\rho_v + (1-x)/\rho_l}$$

where $S$ is the fraction of the volume filled by steam (dimensionless), and $\rho_l$ and $\rho_v$ are the densities of saturated liquid and saturated vapour at the same pressure (kg/m³).

In [ ]:
def steam_fraction(h, P):
    '''Steam mass fraction x (-) for enthalpy h (kJ/kg) at pressure P (MPa).
    Values below 0 mean compressed liquid, above 1 mean superheated steam.'''
    s = saturation(P)
    return ___


def volume_fraction(x, P):
    '''Steam volume fraction S (-) for steam mass fraction x (-) at pressure P (MPa).'''
    s = saturation(P)
    return ___


print(f"x for h = 1400 kJ/kg at 1 MPa: {steam_fraction(1400, 1.0):.3f}")
print(f"S for x = 0.01 at 1 MPa: {volume_fraction(0.01, 1.0):.3f}")

In [ ]:
# Test
assert abs(steam_fraction(1400, 1.0) - 0.316) < 0.003
assert abs(volume_fraction(0.01, 1.0) - 0.635) < 0.005
print("B2.3 OK")

### B2.4 Adiabatic decompression of rising fluid

Fluid rising in a well loses pressure quickly and has little time to exchange heat with the rock around it, and no work is taken out of it. The usual first approximation is that its enthalpy does not change on the way up, so its state moves straight down a vertical line on the $P$-$h$ diagram. Everything else, the temperature, the steam fraction and the density, follows from the two variables $P$ and $h$.

Write a function that returns the state of water of a given enthalpy at a given pressure. Inside the two-phase region use the lever rule from B2.3 and the fact that the temperature is the saturation temperature; outside it, let IAPWS-IF97 do the work.

In [ ]:
def flash(h, P):
    '''State of water of specific enthalpy h (kJ/kg) at pressure P (MPa).
    Returns T (C), steam mass fraction x (-), density (kg/m3) and internal energy u (kJ/kg).'''
    s = saturation(P)
    x = steam_fraction(h, P)                  # from B2.3; outside 0 to 1 means single phase
    if x <= 0 or x >= 1:
        w = IAPWS97(P=P, h=h)                 # compressed liquid or superheated steam
        return {"T": w.T - K0, "x": min(max(x, 0.0), 1.0), "rho": w.rho, "u": w.u}
    v = ___                                   # specific volume of the mixture, m3/kg
    u = ___                                   # internal energy, kJ/kg   (h = u + P v, P in kPa)
    return {"T": s["T"], "x": x, "rho": 1 / v, "u": u}


print(flash(1287.7, 10.0))
print(flash(1287.7, 1.0))

In [ ]:
# Test
st = flash(1287.7, 1.0)
assert abs(st["x"] - 0.261) < 0.005, "steam mass fraction at 1 MPa"
assert abs(st["T"] - 179.9) < 0.5, "inside the dome the temperature is T_sat(P)"
assert abs(st["u"] - 1236) < 3, "u = h - P v, keep MPa and kJ consistent"
assert flash(1287.7, 10.0)["x"] == 0.0
print("B2.4 OK")

Now march the pressure down from the feed zone to the wellhead at constant enthalpy and look at the state along the way.

In [ ]:
# A. Run and look
H_FEED = water(10.0, 290)["h"]                       # kJ/kg, enthalpy of the feed zone fluid
P_path = np.linspace(10.0, 0.5, 97)                  # MPa, pressure on the way up
dec = pd.DataFrame([flash(H_FEED, P) for P in P_path], index=P_path)
dec["Pv"] = P_path * 1e3 / dec["rho"]                # kJ/kg, the flow work term
dec["h"] = H_FEED                                    # kJ/kg, unchanged by assumption

# flashing begins when the pressure reaches the saturation pressure of the feed temperature
P_flash = IAPWS97(T=290 + K0, x=0).P                 # MPa
print(f"feed enthalpy {H_FEED:.0f} kJ/kg, flashing starts at {P_flash:.2f} MPa")

marks = [10.0, 7.4, 5.0, 3.0, 1.0, 0.5]
table = pd.DataFrame([flash(H_FEED, P) for P in marks], index=marks)
table["Pv"] = [P * 1e3 / r for P, r in zip(marks, table["rho"])]
table["u + Pv"] = table["u"] + table["Pv"]
table.index.name = "P (MPa)"
display(table.round(1))

In [ ]:
# A. Run and look
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

ax[0].plot(dec.index, dec["T"], color="C3")
ax[0].set(xlabel="Pressure (MPa)", ylabel="Temperature (C)", title="temperature")

ax[1].plot(dec.index, dec["x"], color="C0")
ax[1].set(xlabel="Pressure (MPa)", ylabel="Steam mass fraction x (-)", title="steam by mass")

# plot the changes since the feed zone, because the sum moves by nothing at all
ax[2].plot(dec.index, dec["u"] - dec["u"].iloc[0], label="change in u")
ax[2].plot(dec.index, dec["Pv"] - dec["Pv"].iloc[0], label="change in P v")
ax[2].plot(dec.index, dec["h"] - dec["h"].iloc[0], color="k", ls="--", label="change in h")
ax[2].set(xlabel="Pressure (MPa)", ylabel="Change since the feed zone (kJ/kg)",
          title="where the energy goes")
ax[2].legend()

for a in ax:
    a.invert_xaxis()                                 # fluid moves left to right as it rises
    a.axvline(P_flash, color="0.6", lw=1, ls=":")
fig.tight_layout()
plt.show()

Three things to take from the figure.

1. The temperature is flat until flashing starts, then it falls along the saturation curve, because inside the two-phase region the temperature is fixed by the pressure.
2. The steam fraction grows from zero at the flash point to about a quarter at 1 MPa. No heat was added. Lowering the pressure moved the equilibrium state until the enthalpy the fluid already had was enough for part of the liquid to be vapour.
3. The third panel shows the change since the feed zone. The internal energy falls and the flow work $Pv$ rises by the same amount, so their sum does not move. The drop in $u$ is the latent heat of the steam that formed. The rise in $Pv$ happens even though the pressure is falling, because the specific volume grows by a factor of about fifty.

**This is an approximation, and a common one.** Three things were left out.

- Heat exchange with the casing and the rock. Over the length of a producing well this is small compared with 1288 kJ/kg, but it is not zero, and in a slow or recently drilled well it matters.
- The elevation and velocity terms. Lifting one kilogram 1000 m costs about 9.8 kJ/kg, under 1 % of the enthalpy here, and the kinetic energy at 50 m/s is about 1.3 kJ/kg. Both are usually dropped.
- Boiling inside the rock is not isenthalpic at all. There the fluid is in contact with a mass of hot rock that holds far more heat than the fluid itself, so the rock feeds the boiling and the enthalpy of the fluid rises as the pressure falls. That is one reason the produced enthalpy of a well can climb over years of production. Isenthalpic decompression is a statement about fluid in a well, not about a reservoir.

### C2. Extensions

1. Plot $S$ against $x$ at 0.5, 5 and 20 MPa. Explain the shape in one sentence using the densities.
2. Compare $u$, $P/\rho$ and $h$ for liquid at 20 °C and 0.1 MPa, liquid at 250 °C and 10 MPa, and steam at 400 °C and 1 MPa. For which state is the flow work a large part of $h$?
3. Liquid water at 300 °C and 10 MPa. Compare density and viscosity with water at 20 °C and 0.1 MPa. Which ratio matters for buoyancy, and which for how easily fluid moves through rock?
4. Plot $c_p$ from 300 to 600 °C at 23, 25 and 30 MPa. What happens near the critical point, and why does it matter for superhot wells?

---
**Break.** Parts 3 and 4 follow the board session on pressure, ascent and flow.

## Part 3. Pressure with depth and the boiling point with depth

### Hydrostatic pressure and the boiling point with depth (BPD)

$$\frac{dP}{dz} = \rho(P,T)\,g, \qquad P_{i+1} = P_i + \rho(P_i, T_i)\,g\,\Delta z$$

where $z$ is depth below the water table (m, positive downward), $\rho(P,T)$ is density at the local pressure and temperature (kg/m³), $i$ is the index of a depth level and $\Delta z$ is the depth step (m).

For the BPD curve, water is at its boiling point at every depth:

$$T_i = T_{sat}(P_i), \qquad \rho_i = \rho_l^{sat}(P_i)$$

where $T_{sat}(P)$ is the boiling temperature at pressure $P$ and $\rho_l^{sat}(P)$ is the density of saturated liquid at $P$.

### B3.1 The BPD curve by stepping down

Watch the units. $\rho g \Delta z$ is in Pa, and our pressure is in MPa. Stop when the pressure reaches the critical pressure.

In [ ]:
def bpd_curve(dz=1.0, z_max=4000.0, P_top=P_ATM, z_top=0.0):
    '''Boiling point with depth for pure water.
    dz: depth step (m).  z_max: maximum depth (m).
    P_top: pressure (MPa) at depth z_top (m), for example the water table.
    Returns arrays z (m), P (MPa), T (C).'''
    z_list, P_list = [z_top], [P_top]
    while z_list[-1] < z_max:
        rho_l = ___                                  # kg/m3, saturated liquid at the current pressure
        P_next = ___                                 # MPa
        if P_next >= P_CRIT:
            break
        z_list.append(z_list[-1] + dz)
        P_list.append(P_next)
    P_arr = np.array(P_list)
    T_arr = np.array([saturation(p)["T"] for p in P_arr])
    return np.array(z_list), P_arr, T_arr


z_b, P_b, T_b = bpd_curve(dz=5.0)
for d in [500, 1000, 2000]:
    print(f"{d:5d} m: P = {np.interp(d, z_b, P_b):6.2f} MPa, T = {np.interp(d, z_b, T_b):5.1f} C")

In [ ]:
# Test
assert abs(np.interp(1000, z_b, P_b) - 7.98) < 0.05
assert abs(np.interp(1000, z_b, T_b) - 294.9) < 1.0
print("B3.1 OK")

### A. Compare your BPD curve with a simulated reservoir

The file `sim_profile_upflow.csv` is a vertical profile taken through the upflow zone of a CSMP++ simulation of a cross-section through the Reykjanes peninsula (Svartsengi sills, pure water, 40 000 years after the first sill). It contains

| column | meaning |
|---|---|
| `depth_m` | depth below the ground surface (m) |
| `T_C` | temperature (°C) |
| `P_MPa` | fluid pressure (MPa) |
| `h_kJ_per_kg` | fluid specific enthalpy (kJ/kg) |
| `steam_saturation` | steam volume fraction (-) |

In [ ]:
URL = "https://raw.githubusercontent.com/two-phaseflow/JED116F/main/data/sim_profile_upflow.csv"
sim = None
try:
    sim = pd.read_csv(URL)                           # normal case: straight from the course repository
except Exception as err:
    print("could not read from GitHub:", err)
    try:
        sim = pd.read_csv("sim_profile_upflow.csv")  # fallback: a copy sitting next to the notebook
    except Exception as err2:
        print("no local copy either:", err2)
        print("Skip this comparison and carry on. Nothing later in the notebook depends on it.")
if sim is not None:
    display(sim.head())

In [ ]:
if sim is None:
    print("no simulation profile loaded, nothing to plot")
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
    a1.plot(sim.T_C, sim.depth_m, color="tab:orange", lw=3, label="simulation")
    a1.plot(T_b, z_b, "k--", lw=2, label="your BPD curve")
    a1.set_xlabel("temperature (C)"); a1.set_ylabel("depth (m)"); a1.set_xlim(0, 400)
    a1.set_ylim(3000, 0); a1.legend()

    a2.plot(sim.P_MPa, sim.depth_m, color="tab:orange", lw=3, label="simulation")
    a2.plot(P_b, z_b, "k--", lw=2, label="BPD column")
    a2.plot(0.1 + 1000 * G * z_b / 1e6, z_b, "b:", lw=2, label="cold hydrostatic")
    a2.set_xlabel("fluid pressure (MPa)"); a2.set_xlim(0, 32); a2.legend()
    plt.show()

**Questions to answer in one or two sentences each.**

1. Over which depth range does the simulated temperature follow the BPD curve?
2. What happens below that range, and what does it tell you about how the fluid is moving there?
3. The simulated pressure is close to cold hydrostatic even though the fluid is hot. What does that say about where the pressure in the column comes from?

### C3. Extensions

1. **Step size.** Run `bpd_curve` with `dz` = 200, 100, 50, 10 and 1 m. How much does the pressure at 1000 m change?
2. **Water table.** Move the water table to 200 m depth by starting the march there with `P_top = P_ATM` and `z_top = 200`. How does the BPD temperature at 1000 m change?
3. **Above the curve.** A well measures 320 °C in liquid at 1000 m, where your BPD curve gives 295 °C. List two physical explanations.

## Before you leave. Reflection

Answer in one or two sentences each.

1. Write the equation you solved in Part 3 and define every symbol with its unit.
2. Name two assumptions behind the BPD curve.
3. Name one way this simple model could mislead someone planning a well.